In [1]:
# Import libraries here

import numpy as np
import pandas as pd
import itertools
import os
import sys
import scipy.optimize as sco

In [2]:
current_dir = os.path.abspath('')
project_root = os.path.abspath(os.path.join(current_dir, '../../'))

if project_root not in sys.path:
    sys.path.append(project_root)

os.chdir(project_root)

print(f"Working directory set to: {os.getcwd()}")

Working directory set to: D:\users\kamen.dimitrov\desktop\softuni\math_concepts_for_developers\08_final_project


In [3]:
# Import project modules here

import importlib
from src.portfolio_simulation_utils import portfolio_simulation as p_sim
from src.data_pipeline_utils import data_fetching_handling as data_pipe
importlib.reload(p_sim)
importlib.reload(data_pipe)

<module 'src.data_pipeline_utils.data_fetching_handling' from 'D:\\users\\kamen.dimitrov\\desktop\\softuni\\math_concepts_for_developers\\08_final_project\\src\\data_pipeline_utils\\data_fetching_handling.py'>

## Harry Markowitz Efficient Frontier ##

Let us now repeat the exercise with the same stocks, however using more robust mathematical modeling skills. While I am building this project, I am very curious if applying the mathematical concepts more carefully will lead to a result, which is similar enough to the practical approach I have utilized in the past. 

The first step is the use the exact same tickers and rebuild our returns dataframe, very similar to the practical approach

In [4]:
tickers = ['AAPL', 'NVDA', 'MSFT', 'JNJ', 'BAC', 'VZ', 'WMT', 'UPS', 'PFE', 'JPM']

In [5]:
returns_df =  data_pipe.build_returns_df(tickers)

### TODO -> add description ###

1. Compute the Input Matrices (Data Pipeline)Your current pipeline fetches historical prices. You must convert these into a mean returns vector and a covariance matrix. These are the fundamental inputs for the analytical approach.Expected Returns Vector ($\mu$): Calculate the annualized mean of the daily returns for each asset.Covariance Matrix ($\Sigma$): Calculate the annualized covariance of the daily returns between all asset pairs.

In [6]:
trading_days = 252
mean_returns = returns_df.mean() * trading_days
cov_matrix = returns_df.cov() * trading_days

In [7]:
print(mean_returns)

AAPL    0.252990
NVDA    0.549934
MSFT    0.220036
JNJ     0.111716
BAC     0.165342
VZ      0.049119
WMT     0.191497
UPS     0.056215
PFE     0.040092
JPM     0.196708
dtype: float64


In [8]:
print(cov_matrix)

          AAPL      NVDA      MSFT       JNJ       BAC        VZ       WMT  \
AAPL  0.083463  0.077246  0.051357  0.015957  0.037046  0.011788  0.019645   
NVDA  0.077246  0.244553  0.081753  0.009701  0.049861  0.005066  0.021721   
MSFT  0.051357  0.081753  0.072296  0.014825  0.034087  0.010460  0.018207   
JNJ   0.015957  0.009701  0.014825  0.033575  0.017611  0.014202  0.012813   
BAC   0.037046  0.049861  0.034087  0.017611  0.094832  0.017849  0.015987   
VZ    0.011788  0.005066  0.010460  0.014202  0.017849  0.040717  0.012009   
WMT   0.019645  0.021721  0.018207  0.012813  0.015987  0.012009  0.046576   
UPS   0.033177  0.042817  0.028749  0.015029  0.036421  0.014347  0.016428   
PFE   0.019719  0.016842  0.018335  0.022105  0.023427  0.015801  0.012307   
JPM   0.032794  0.045553  0.031207  0.016829  0.074396  0.015607  0.014115   

           UPS       PFE       JPM  
AAPL  0.033177  0.019719  0.032794  
NVDA  0.042817  0.016842  0.045553  
MSFT  0.028749  0.018335  0.03

### 2. Define the Linear Algebra  ##

Modern Portfolio Theory evaluates portfolios using matrix multiplication. You must define functions to compute portfolio performance given a weight vector ($w$).

Portfolio Return: $E(R_p) = w^T \mu$

Portfolio Variance: $\sigma_p^2 = w^T \Sigma w$

Let us define the starting weights vector by stating an equal weight of each stock in the portfolio for a starting point

In [9]:
number_of_stocks = len(tickers)
weights = np.array([1 / number_of_stocks] * number_of_stocks)

In [10]:
def calc_portfolio_performance(weights, mean_returns, cov_matrix):
    returns = np.dot(weights, mean_returns)
    std_dev = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return returns, std_dev

returns, std_dev = calc_portfolio_performance(weights, mean_returns, cov_matrix)

In [11]:
print(returns)
print(std_dev)

0.18336492527053894
0.1776577128664036


In [12]:
def negative_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate=0.03):
    p_ret, p_std = calc_portfolio_performance(weights, mean_returns, cov_matrix)
    return -(p_ret - risk_free_rate) / p_std

In [13]:
args = (mean_returns, cov_matrix)

# Constraint: sum of weights equals 1
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})

# Bounds: weights between 0 and 1
bounds = tuple((0, 1) for asset in range(number_of_stocks))

# Initial guess (equal distribution)
initial_guess = number_of_stocks * [1. / number_of_stocks]

In [14]:
optimal_portfolio = sco.minimize(
    negative_sharpe_ratio, 
    initial_guess, 
    args=args,
    method='SLSQP', 
    bounds=bounds, 
    constraints=constraints
)

optimal_weights = optimal_portfolio.x
optimal_return = np.sum(mean_returns * optimal_weights)
optimal_variance = np.dot(optimal_weights.T, np.dot(cov_matrix, optimal_weights))
optimal_sharpe_ratio = optimal_return / np.sqrt(optimal_variance)

print(f"Expected Return: {optimal_return:.4f}")
print(f"Variance: {optimal_variance:.4f}")

Expected Return: 0.2992
Variance: 0.0499


In [15]:
print(list(zip(tickers, optimal_weights)))

[('AAPL', np.float64(0.03702352719510818)), ('NVDA', np.float64(0.3234816059369963)), ('MSFT', np.float64(0.0)), ('JNJ', np.float64(0.13740432059348773)), ('BAC', np.float64(0.0)), ('VZ', np.float64(0.0)), ('WMT', np.float64(0.4104997164522972)), ('UPS', np.float64(1.702197410802242e-17)), ('PFE', np.float64(0.0)), ('JPM', np.float64(0.09159082982211066))]


### Sanity Check ###

We can take the optimal portfolio using the algorithmic solution from above in a list and the optimal portfolio produced by the portfolio simulation done with Monte Carlo methods. As said, the great the number of sim runs is, the closer the result is. Unfortunately my machine takes nearly 20 minutes to do 30 000 runs and I haven't experimented with a greater number of runs. 

In [16]:
list_1 = [('AAPL', np.float64(0.03702352719510818)), ('NVDA', np.float64(0.3234816059369963)), ('MSFT', np.float64(0.0)), ('JNJ', np.float64(0.13740432059348773)), ('BAC', np.float64(0.0)), ('VZ', np.float64(0.0)), ('WMT', np.float64(0.4104997164522972)), ('UPS', np.float64(1.702197410802242e-17)), ('PFE', np.float64(0.0)), ('JPM', np.float64(0.09159082982211066))]
list_2 = [('AAPL', np.float64(0.03702352719510818)), ('NVDA', np.float64(0.3234816059369963)), ('MSFT', np.float64(0.0)), ('JNJ', np.float64(0.13740432059348773)), ('BAC', np.float64(0.0)), ('VZ', np.float64(0.0)), ('WMT', np.float64(0.4104997164522972)), ('UPS', np.float64(1.702197410802242e-17)), ('PFE', np.float64(0.0)), ('JPM', np.float64(0.09159082982211066))]
for i in range(len(tickers)):
    print(f"Ticker {list_1[i][0]} -> Monte Carlo {list_1[i][1] * 100:.2f} - Math algorithm {list_2[i][1] * 100:.2f}"
    f"\nDifference {(list_1[i][1] * 100 - list_2[i][1] * 100):.2f}")

Ticker AAPL -> Monte Carlo 3.70 - Math algorithm 3.70
Difference 0.00
Ticker NVDA -> Monte Carlo 32.35 - Math algorithm 32.35
Difference 0.00
Ticker MSFT -> Monte Carlo 0.00 - Math algorithm 0.00
Difference 0.00
Ticker JNJ -> Monte Carlo 13.74 - Math algorithm 13.74
Difference 0.00
Ticker BAC -> Monte Carlo 0.00 - Math algorithm 0.00
Difference 0.00
Ticker VZ -> Monte Carlo 0.00 - Math algorithm 0.00
Difference 0.00
Ticker WMT -> Monte Carlo 41.05 - Math algorithm 41.05
Difference 0.00
Ticker UPS -> Monte Carlo 0.00 - Math algorithm 0.00
Difference 0.00
Ticker PFE -> Monte Carlo 0.00 - Math algorithm 0.00
Difference 0.00
Ticker JPM -> Monte Carlo 9.16 - Math algorithm 9.16
Difference 0.00


## Combinations

Let's experiment with Combinations to make the study more exciting. It is going to be informative to find all possible portfolios of five stocks based on the ten stocks we have chosen. 

To find all possible combinations of 5 assets from a pool of 10, you use the binomial coefficient (combinations). The formula is:
$$\binom{n}{k} = \frac{n!}{k!(n-k)!}$$

If we run the formula, we find out that there are exactly 252 unique ways to select a 5-ticker portfolio from your list of 10.

We use the cool itertools library in Python to make it instantenous in applying this mathematical calculation

In [17]:
portfolio_combinations = list(itertools.combinations(tickers, 5))

print(f"Total portfolios generated: {len(portfolio_combinations)}")
print(portfolio_combinations[0])

Total portfolios generated: 252
('AAPL', 'NVDA', 'MSFT', 'JNJ', 'BAC')


### Let's optimize all 252 portfolios we received ###

Below, we are going to replicate the portfolio optimization technique we applied above.

In [18]:
number_of_stocks = 5
trading_days = 252
portfolio_metrics = {}
counter = 1

for portfolio in portfolio_combinations:

    current_weights = np.array([1 / number_of_stocks] * number_of_stocks)
    tickers = list(portfolio)
    returns_df =  data_pipe.build_returns_df(tickers)
    
    current_mean_returns = returns_df.mean() * trading_days
    current_cov_matrix = returns_df.cov() * trading_days

    returns, std_dev = calc_portfolio_performance(current_weights, current_mean_returns,current_cov_matrix)

    args = (current_mean_returns, current_cov_matrix)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds = tuple((0, 1) for asset in range(number_of_stocks))
    initial_guess = number_of_stocks * [1. / number_of_stocks]

    current_optimal_portfolio = sco.minimize(
    negative_sharpe_ratio, 
    initial_guess, 
    args=args,
    method='SLSQP', 
    bounds=bounds, 
    constraints=constraints
    )

    current_optimal_weights = current_optimal_portfolio.x
    current_return = np.sum(current_mean_returns * current_optimal_weights)
    current_variance = np.dot(current_optimal_weights.T, np.dot(current_cov_matrix, current_optimal_weights))
    current_sharpe = current_return / np.sqrt(current_variance)
    
    portfolio_metrics[f"Portfolio {counter}"] = {'Tickers': tickers,
                                  'Weights': current_optimal_weights, 
                                  'Return': current_return, 
                                  'Variance': current_variance,
                                  'Variance Deviation': current_variance - optimal_variance,
                                    "Sharpe Ratio": current_sharpe,
                                "Sharpe Deviation": optimal_sharpe_ratio - current_sharpe,
                                 }

    counter += 1

print(portfolio_metrics)

{'Portfolio 1': {'Tickers': ['AAPL', 'NVDA', 'MSFT', 'JNJ', 'BAC'], 'Weights': array([1.58670478e-01, 4.53728346e-01, 4.20922958e-17, 3.87601177e-01,
       0.00000000e+00]), 'Return': np.float64(0.33296428395070526), 'Variance': np.float64(0.07398881746966551), 'Variance Deviation': np.float64(0.02408115524738387), 'Sharpe Ratio': np.float64(1.2240935440883196), 'Sharpe Deviation': np.float64(0.11537013724885159)}, 'Portfolio 2': {'Tickers': ['AAPL', 'NVDA', 'MSFT', 'JNJ', 'VZ'], 'Weights': array([1.58197284e-01, 4.53829945e-01, 0.00000000e+00, 3.87972771e-01,
       4.28595803e-17]), 'Return': np.float64(0.33294195660982806), 'Variance': np.float64(0.07397790135540472), 'Variance Deviation': np.float64(0.024070239133123075), 'Sharpe Ratio': np.float64(1.224101764646477), 'Sharpe Deviation': np.float64(0.11536191669069429)}, 'Portfolio 3': {'Tickers': ['AAPL', 'NVDA', 'MSFT', 'JNJ', 'WMT'], 'Weights': array([0.05884275, 0.3404751 , 0.        , 0.17241729, 0.42826486]), 'Return': np.fl

### Let's observe the results of this experiment."

We can see that the majority of the portfolios constructed by a subset of 5 stocks from the originally selected stocks exhibit worse risk characteristics than a portfolio constructed of all selected tickers. Yet, we can find 5-ticker portfolios, which are possible constructed using stocks with lower variance of their returns, i.e., less risky. However, when we compare the utilization of risk to achieve a unit of return, we observe that the optimized portfolio using all selected tickers is expected to be more efficient.  

While individual 5-stock subsets can achieve lower absolute variance by selecting low-volatility assets, the 10-stock universe consistently provides a superior Risk-Adjusted Return. The 10-stock (in our case) optimal portfolio maintains a Sharpe Ratio that is, on average, 0.20 to 0.60 points higher than 5-stock combinations, proving that a broader asset base allows for a more efficient frontier.

In [22]:
for portfolio, metric in portfolio_metrics.items():
    print(f"{portfolio} -> Variance Deviation: {metric['Variance Deviation'] * 100:.2f}, Sharpe Ratio Deviation: {metric['Sharpe Deviation']:.2f}")

Portfolio 1 -> Variance Deviation: 2.41, Sharpe Ratio Deviation: 0.12
Portfolio 2 -> Variance Deviation: 2.41, Sharpe Ratio Deviation: 0.12
Portfolio 3 -> Variance Deviation: 0.20, Sharpe Ratio Deviation: 0.01
Portfolio 4 -> Variance Deviation: 2.41, Sharpe Ratio Deviation: 0.12
Portfolio 5 -> Variance Deviation: 2.40, Sharpe Ratio Deviation: 0.12
Portfolio 6 -> Variance Deviation: 1.89, Sharpe Ratio Deviation: 0.10
Portfolio 7 -> Variance Deviation: 9.18, Sharpe Ratio Deviation: 0.18
Portfolio 8 -> Variance Deviation: 1.48, Sharpe Ratio Deviation: 0.03
Portfolio 9 -> Variance Deviation: 9.20, Sharpe Ratio Deviation: 0.18
Portfolio 10 -> Variance Deviation: 9.19, Sharpe Ratio Deviation: 0.18
Portfolio 11 -> Variance Deviation: 5.98, Sharpe Ratio Deviation: 0.15
Portfolio 12 -> Variance Deviation: 1.48, Sharpe Ratio Deviation: 0.03
Portfolio 13 -> Variance Deviation: 9.77, Sharpe Ratio Deviation: 0.18
Portfolio 14 -> Variance Deviation: 9.78, Sharpe Ratio Deviation: 0.18
Portfolio 15 ->

## Conclusion